# QEC code → QCCD simulator: end-to-end walkthrough

目的: 移動無し native 形式の QEC 回路 (= `circuits/<code>.py`) を入口に、
QCCD コンパイラ・Stim 再変換・ノイズ挿入・MWPM 復号 を一通り辿って **論理
誤り率 $p_L$ を取得** するまでの workflow を示す。

## 想定読者
新たな QEC 符号 (今回の例は surface code / repetition code 以外) で同じ
シミュレーションを試したい人。 `circuits/<新しい code>.py` を 1 つ用意する
だけで、 このノートブックの他セルはほぼ無変更で再利用できる。

## 全体像
```
Stim 回路 (or 論文の stabilizer 表) 
       │   ※ ここは手作業: stabilizer-explicit + ancilla-reuse 化
       ↓
circuits/<code>.py  (移動無し native DSL)   ← 入口
       │                                       本ノートブックで自動化される範囲
       ├─ Step 1: parse → DAG
       ├─ Step 2: architecture + placement
       ├─ Step 3: QCCD compile → flat_ops + timesteps + phonon track
       ├─ Step 4: recompile to Stim (DETECTOR + OBSERVABLE 付き)
       ├─ Step 5: ノイズ注入 (PAULI_CHANNEL_2 lookup + transport dephasing)
       └─ Step 6: MWPM 復号 → 論理誤り率 p_L
                                              ← 出口
```

## 入口の制約 (重要)
`circuits/<code>.py` は以下を満たす必要がある:
- **stabilizer 境界が明示**: `RESET ancilla` → R/MS gates → `MEASURE ancilla` の
  ブロックが各 stabilizer に対応する形
- **ancilla reuse**: 1 つの ancilla を順番に使い回し (Stim 標準テンプレート
  のように stab ごとに新規 ancilla を取る形式は **非対応**)
- **serial**: 1 行 1 命令 (並列化は compiler が決める)
- フォーマットは `examples/single_liz/d3_reuse1/...` 等の既存 circuit を参考に
  ([`circuits/d3_rep_reuse1.py`](../circuits/d3_rep_reuse1.py) が最小例)

In [ ]:
# ==========================================
# Imports + paths
# ==========================================
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
for sub in ('src', 'circuits', 'experiments'):
    sys.path.insert(0, str(REPO / sub))

import numpy as np
import math
import stim
import pymatching

from qccd_parser import parse
from dag_builder import DAGBuilder
from qccd_compiler import QCCDArchitecture, QCCDCompiler, CompilerConfig
from stim_recompiler import recompile_multi_round
from pauli_lookup import PauliErrorLookup
from exp_noise_injection import (
    phonon_to_pauli_lookup_rates,
    inject_noise_approach_b,
    inject_transport_dephasing,
    default_noise_params,
)
from exp_multi_round import placement_greedy_alt

PAULI_LOOKUP = PauliErrorLookup()
print('Setup OK. Repo root:', REPO)

## Step 0: 入口の circuit module を選ぶ

ここで使う符号を指定。 既存は以下:
- `'snake_reuse1'` (d=3 rotated surface code) — `'snake_reuse1_reverse'`
- `'d5_reuse1'` (d=5), `'d5_reuse1_reverse_cnot'`
- `'d7_reuse1'` (d=7), `'d7_reuse1_reverse_cnot'`
- `'d9_reuse1'` (d=9), `'d9_reuse1_reverse_cnot'`
- `'d3_rep_reuse1'` (d=3 repetition code) — reverse 無し (1 ラウンド前提)

新しい符号を追加する場合:
1. `circuits/<name>.py` を作成 (上記制約に従う)
2. 同形式の reverse 版を作るか、 multi-round を 1 ラウンドに留める
3. 下の `FORWARD_NAME` を変更

In [ ]:
# このノートブック例: snake_reuse1 (d=3 surface code)
FORWARD_NAME = 'snake_reuse1'
REVERSE_NAME = 'snake_reuse1_reverse'    # multi-round の偶数ラウンドで使う
OBSERVABLE_QUBITS = [0, 1, 2]            # 論理 Z の構成 qubit (top row)
ROUNDS = 3                                # 何ラウンド回すか

fwd_mod = __import__(FORWARD_NAME)
print(f'circuit:        {FORWARD_NAME}')
print(f'data qubits:    {fwd_mod.DATA_QUBITS}')
print(f'ancilla qubits: {fwd_mod.ANCILLA_QUBITS}')
print(f'CIRCUIT (head):')
print('\n'.join(fwd_mod.CIRCUIT.strip().splitlines()[:15]))

## Step 1-3: parse → DAG → compile (1 ラウンド)

compile は以下を出力する:
- `flat_ops`: native 命令の平坦列 (Split / Merge / Shuttle / Swap / R / MS / Reset / Measure)
- `timesteps`: 並列実行を考慮した scheduling 結果
- `phonon_data`: 各 MS gate 時点のフォノン数 (axial / transversal、 ion ごと)

アーキテクチャは `T-M-T-L-T-M-T` (LIZ 1 個 + Memory 2 個 + その他 Temp)。 placement
は `greedy_alt` (回路の登場順に LIZ 近接 zone から左右交互、 canonical)。

In [ ]:
def compile_round(mod_name, placement, arch, initial_zone_contents=None):
    """1 ラウンドのコンパイル. multi-round 用に zone_contents で順序を引き継ぐ."""
    mod = __import__(mod_name)
    program = parse(mod.CIRCUIT)
    dag = DAGBuilder(program, stabilizer_parallel=False).build()
    if initial_zone_contents is not None:
        ordered = {}
        for zid in sorted(initial_zone_contents.keys()):
            for ion_id in initial_zone_contents[zid]:
                ordered[ion_id] = zid
        placement = ordered
    compiler = QCCDCompiler(arch, config=CompilerConfig())
    return compiler.compile(dag, placement)

# Architecture: simple_mlm (T-M-T-L-T-M-T), capacity は data 数に合わせて調整
half = (len(fwd_mod.DATA_QUBITS) + 1) // 2
mem_cap = max(20, half + len(fwd_mod.ANCILLA_QUBITS) + 4)
arch = QCCDArchitecture.simple_mlm(liz_capacity=4, memory_capacity=mem_cap)

# Placement: greedy_alt (canonical)
initial = placement_greedy_alt(fwd_mod.DATA_QUBITS, fwd_mod.ANCILLA_QUBITS, fwd_mod)
print(f'initial placement: {initial}')

# Multi-round compile (forward/reverse 交互)
results = []
current_placement = initial
current_zone_contents = None
for r in range(1, ROUNDS + 1):
    mod_name = FORWARD_NAME if r % 2 == 1 else REVERSE_NAME
    result = compile_round(mod_name, current_placement, arch,
                           initial_zone_contents=current_zone_contents)
    results.append(result)
    current_placement = result.final_placement
    current_zone_contents = result.final_zone_contents
    print(f'round {r} ({mod_name}): {len(result.flat_ops)} ops, '
          f'{len(result.timesteps)} timesteps, phonon tracking '
          f'{"on" if result.phonon_data else "off"}')

### (任意) compile の中身を覗く

`flat_ops.txt` / `timesteps.txt` の表記は `examples/native_instruction_format.md`
で詳細解説してある。 ここでは round 1 の最初 10 命令だけ summary。

In [ ]:
from formatter import format_op_summary
for i, op in enumerate(results[0].flat_ops[:10]):
    print(f'{i:3d}: {format_op_summary(op)}')

## Step 4: recompile to Stim (clean logical 回路)

compile 結果 → Stim の **logical-level 回路** (R/H/CX/M) に変換。 DETECTOR と
OBSERVABLE_INCLUDE まで自動で挿入される。 ノイズはまだ無い。

In [ ]:
clean_circuit = recompile_multi_round(results, observable_column=OBSERVABLE_QUBITS)
n_inst = sum(1 for _ in clean_circuit)
print(f'clean Stim circuit: {n_inst} instructions, {clean_circuit.num_qubits} qubits')

# noiseless sample (= 全 detector が 0、 観測値も 0 になるはず)
sampler = clean_circuit.compile_detector_sampler(seed=42)
det, obs = sampler.sample(100, separate_observables=True)
print(f'noiseless 100 shots → detectors firing: {int(det.sum())}, '
      f'observable flips: {int(obs.sum())} (両方 0 ならOK)')

## Step 5: ノイズ注入

3 段階で挿入:
1. **MS ゲートエラー**: PAULI_CHANNEL_2 (lookup table 由来の 15 Pauli 確率)
2. **その他のゲートエラー**: DEPOLARIZE1 (R), X_ERROR (RESET/MEASURE), idle Z_ERROR
3. **Transport-time dephasing**: ラウンドごとに Z_ERROR (lumped)

MS ゲートエラーの計算には **heating モード** が 3 種類:
- `phonon_scale=1.0`: 実 phonon をそのまま使用 (canonical)
- `phonon_scale=α (0<α<1)`: heating を α 倍に抑えた仮想ハードウェア
- `override_n_bar=0.01`: 全 MS で固定 n_bar=0.01 (heating-free 限界)

下のセルで `PHONON_SCALE` / `NO_HEATING` を切り替えると 3 シナリオを試せる。

In [ ]:
PHONON_SCALE = 1.0     # MS 時点の transversal phonon に掛ける倍率
NO_HEATING = False     # True にすると override_n_bar=0.01 で固定

# phonon_at_gates を全 round 分集める
all_phonon_gates = []
for r in results:
    pd = r.phonon_data or {}
    all_phonon_gates.extend(pd.get('at_gates', []))
print(f'total MS gates with phonon record: {len(all_phonon_gates)}')

# PAULI_CHANNEL_2 lookup
pauli_rates = phonon_to_pauli_lookup_rates(
    all_phonon_gates, PAULI_LOOKUP, mode='strict',
    override_n_bar=(0.01 if NO_HEATING else None),
    phonon_scale=PHONON_SCALE,
)

# MS ノイズの sum (15 Pauli 和) の分布を覗く
sum_p = [sum(r[1]) for r in pauli_rates]
print(f'MS PAULI_CHANNEL_2 prob sum: min={min(sum_p):.3e}, '
      f'max={max(sum_p):.3e}, mean={sum(sum_p)/len(sum_p):.3e}')

# ノイズ挿入
noise_params = default_noise_params()
noisy = inject_noise_approach_b(clean_circuit, pauli_rates, noise_params)
noisy = inject_transport_dephasing(noisy, results, noise_params)
print(f'noisy Stim circuit: {sum(1 for _ in noisy)} instructions')

### (任意) noisy 回路の中身確認

Stim ファイルとして書き出しておくと、 そのまま `stim.Circuit('...')` で誰でも
再現できる形になる。 ここでは保存場所だけ示す:

In [ ]:
OUT_DIR = REPO / 'results' / 'walkthrough'
OUT_DIR.mkdir(parents=True, exist_ok=True)
stim_path = OUT_DIR / f'noisy_{FORWARD_NAME}_{ROUNDS}r.stim'
stim_path.write_text(str(noisy), encoding='utf-8')
print(f'saved: {stim_path.relative_to(REPO)}')
print('--- 最初の 10 命令 ---')
for i, inst in enumerate(noisy):
    if i >= 10: break
    print(f'  {inst}')

## Step 6: MWPM 復号 → p_L

Stim の detector_error_model を pymatching に渡して minimum-weight perfect
matching で復号、 観測値の不一致率を $p_L$ として返す。

`approximate_disjoint_errors=True` は `PAULI_CHANNEL_2` を含む回路で必須
(15 Pauli を disjoint と近似する)。

In [ ]:
SHOTS = 50_000   # 必要に応じて増減。 p_L が小さい時は 100k 〜 1M

sampler = noisy.compile_detector_sampler(seed=42)
dem = noisy.detector_error_model(decompose_errors=True,
                                  approximate_disjoint_errors=True)
matcher = pymatching.Matching.from_detector_error_model(dem)

det_samples, obs_actual = sampler.sample(SHOTS, separate_observables=True)
obs_predicted = matcher.decode_batch(det_samples)
errors = int(np.any(obs_predicted != obs_actual, axis=1).sum())

p_L = errors / SHOTS
stderr = math.sqrt(p_L * (1 - p_L) / SHOTS) if SHOTS > 0 else 0
per_round = 1 - (1 - p_L) ** (1.0 / ROUNDS) if p_L > 0 else 0

print(f'=== MWPM result ===')
print(f'  shots:       {SHOTS}')
print(f'  errors:      {errors}')
print(f'  p_L:         {p_L:.4e} ± {stderr:.4e}  (over {ROUNDS} rounds)')
print(f'  per-round:   {per_round:.4e}')

## Step 7 (任意): 複数 d を sweep して plot

既存の `exp_distance_scan.py` がこの sweep を CLI 1 発で回す。
Notebook 内で同じことをしたい場合は以下のループ。
新規符号で同様に作る場合は `DISTANCE_CIRCUITS` 相当の dict を組めば良い。

In [ ]:
# Surface code 系の例 (heating-free モードで d=3,5,7)
import matplotlib.pyplot as plt

DISTANCE_CIRCUITS = {
    3: ('snake_reuse1',  'snake_reuse1_reverse',         [0, 1, 2]),
    5: ('d5_reuse1',     'd5_reuse1_reverse_cnot',       [0, 1, 2, 3, 4]),
    7: ('d7_reuse1',     'd7_reuse1_reverse_cnot',       [0, 1, 2, 3, 4, 5, 6]),
}

def run_one(d, fwd, rev, obs, rounds=3, shots=50_000, no_heating=True):
    fwd_mod = __import__(fwd)
    half = (len(fwd_mod.DATA_QUBITS) + 1) // 2
    mem_cap = max(20, half + len(fwd_mod.ANCILLA_QUBITS) + 4)
    arch = QCCDArchitecture.simple_mlm(liz_capacity=4, memory_capacity=mem_cap)
    initial = placement_greedy_alt(fwd_mod.DATA_QUBITS, fwd_mod.ANCILLA_QUBITS, fwd_mod)

    res = []
    p, zc = initial, None
    for r in range(1, rounds + 1):
        mod_name = fwd if r % 2 == 1 else rev
        rr = compile_round(mod_name, p, arch, initial_zone_contents=zc)
        res.append(rr)
        p, zc = rr.final_placement, rr.final_zone_contents

    circuit = recompile_multi_round(res, observable_column=obs)
    gates = []
    for r in res:
        gates.extend((r.phonon_data or {}).get('at_gates', []))
    pauli_rates = phonon_to_pauli_lookup_rates(
        gates, PAULI_LOOKUP, mode='strict',
        override_n_bar=(0.01 if no_heating else None))
    noisy = inject_noise_approach_b(circuit, pauli_rates, default_noise_params())
    noisy = inject_transport_dephasing(noisy, res, default_noise_params())

    sampler = noisy.compile_detector_sampler(seed=42 + d)
    dem = noisy.detector_error_model(decompose_errors=True,
                                     approximate_disjoint_errors=True)
    matcher = pymatching.Matching.from_detector_error_model(dem)
    det, ob_a = sampler.sample(shots, separate_observables=True)
    ob_p = matcher.decode_batch(det)
    errors = int(np.any(ob_a != ob_p, axis=1).sum())
    return errors, shots

scan = []
for d, (fwd, rev, obs) in DISTANCE_CIRCUITS.items():
    e, s = run_one(d, fwd, rev, obs)
    pL = e / s; se = math.sqrt(pL*(1-pL)/s) if s else 0
    print(f'd={d}: errors={e}/{s}, p_L={pL:.3e} ± {se:.3e}')
    scan.append((d, pL, se))

ds = [x[0] for x in scan]
ps = [x[1] for x in scan]
es = [x[2] for x in scan]
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(ds, ps, yerr=es, fmt='o-', markersize=8, capsize=4)
ax.set_yscale('log')
ax.set_xlabel('Code distance d')
ax.set_ylabel('Logical error rate $p_L$')
ax.set_title('error vs distance (heating free)')
ax.grid(alpha=0.3, which='both')
plt.show()

## 新規符号を追加するには

### A. 「移動無し native」 (= `circuits/<code>.py`) を手作りする

**形式の最小例** ([`circuits/d3_rep_reuse1.py`](../circuits/d3_rep_reuse1.py)):

```python
QUBIT_COUNT = 4
DATA_QUBITS = [0, 1, 2]
ANCILLA_QUBITS = [3]

CIRCUIT = """
QUBIT 0 AT (0, 0)
QUBIT 1 AT (0, 1)
...
QUBIT 3 AT (2, 0)

# Stabilizer 1: Z01
RESET 3
R 0 1.5708 1.5708    # pre-rotation on data
MS 0 3 0.25          # MS gate (data, ancilla)
R 0 1.5708 3.1416    # post-rotation 1
R 3 1.5708 3.1416    # post-rotation 2
R 0 1.5708 4.7124    # post-rotation 3
...
MEASURE 3

# Stabilizer 2: ...
RESET 3              # ← ancilla reuse
...
MEASURE 3
"""
```

コンパイラに食わせるための制約:
1. `RESET <ancilla>` → 各種 R/MS → `MEASURE <ancilla>` の **ブロック単位で**
   stabilizer を表現する
2. **ancilla は再利用** (上の例で q3 を使い回している)
3. **MS の target 順**:  Z-type stabilizer は `MS <data> <ancilla>`、
   X-type stabilizer は `MS <ancilla> <data>` (compiler は recompile 時に
   この順番を見て X/Z を判定する)
4. **R rotation の angle**: 既存例の `1.5708 1.5708`, `1.5708 3.1416`,
   `1.5708 4.7124`, `3.1416 0.0` 等の組合せをそのまま流用すれば OK (Mølmer-
   Sørensen 系の trapped-ion CNOT 分解に対応)

### B. observable_column の定め方

符号の **論理 Z** (or X) の表現に従って `observable_column = [...]` を決める。
rotated surface code であれば横方向 row (例 d=3: `[0,1,2]`)、 repetition code
であれば任意の単 qubit (例: `[0]`)。

### C. multi-round 用 reverse 版

3 ラウンド以上回すなら、 stabilizer 順を逆にした `<code>_reverse` も用意する。
DETECTOR は `(basis, frozenset(data_qubits))` で identity match するので、
stabilizer 順序の違いに依存しない。

### D. ノートブック側の変更点

- `FORWARD_NAME`, `REVERSE_NAME`, `OBSERVABLE_QUBITS` を新符号に合わせて更新
- 他のセルは無変更で OK

## 補足: 既存の対応 sub-pipeline

本 notebook の各 Step は CLI script でも同等のことができる。

| Step | notebook | CLI script |
|------|----------|-----------|
| 1-3 (compile) + 4 (recompile) + 5 (noise) | 上記セル | `experiments/exp_build_examples.py` (snapshot 用) |
| 6 (MWPM 単発) | 上記セル | `experiments/exp_stim_simulation.py` |
| 7 (distance scan) | 上記セル | `experiments/exp_distance_scan.py --distances 3 5 7` |
| heating-free / scaled | 上記 PHONON_SCALE | `--no-heating` / `--phonon-scale 0.5` 等 |

結果はそれぞれ `examples/single_liz/`, `examples/distance_scan*/` 等に
保存済み。 比較対象として参照可能。